## 0. 환경설정 — 스드메 통합 챗봇

하나의 채팅방에서 스튜디오/드레스/메이크업 질문을 자유롭게 할 수 있습니다.

In [1]:
!pip install neo4j-graphrag neo4j openai mysql-connector-python gradio python-dotenv -q

In [2]:
import os, json, re, time
from dotenv import load_dotenv

load_dotenv()

from neo4j import GraphDatabase, basic_auth
from neo4j_graphrag.retrievers import Text2CypherRetriever
try:
    from neo4j_graphrag.llm import OpenAILLM
except ImportError:
    from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.generation import GraphRAG, RagTemplate
from openai import OpenAI
import gradio as gr

print("모듈 로드 완료")

c:\Users\SSAFY\miniforge3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


모듈 로드 완료


In [3]:
client = OpenAI()

## 1. Neo4j 연결 (읽기 전용 — 데이터 적재는 db_load.py)

In [4]:
NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://127.0.0.1:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ["NEO4J_PW"]

driver = GraphDatabase.driver(NEO4J_URI, auth=basic_auth(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    cnt = session.run("MATCH (n) RETURN count(n) AS cnt").single()["cnt"]
    print(f"Neo4j 현재 노드 수: {cnt}")

Neo4j 현재 노드 수: 15959


In [5]:
# 카테고리별 데이터 존재 확인
with driver.session() as session:
    for cat in ["studio", "dress", "makeup"]:
        cnt = session.run(
            "MATCH (v:Vendor {category: $cat}) RETURN count(v) AS cnt", cat=cat
        ).single()["cnt"]
        status = f"{cnt}개 확인" if cnt > 0 else "⚠ 0개 — db_load.py 실행 필요"
        print(f"  {cat}: {status}")

  studio: 207개 확인
  dress: 268개 확인
  makeup: 472개 확인


## 2. MySQL (더미 fallback)

In [6]:
import mysql.connector

MYSQL_HOST = os.environ.get("MYSQL_HOST", "")
MYSQL_USER = os.environ.get("MYSQL_USER", "")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD", "")
MYSQL_DB = os.environ.get("MYSQL_DB", "")
MYSQL_PORT = int(os.environ.get("MYSQL_PORT", 3306))
COUPLE_ID = 2

DUMMY_PREF = {"region": "서울", "sub_region": "강남구",
              "studio_style": "인물중심", "dress_style": "클래식", "makeup_style": "내추럴"}
DUMMY_LIKES = [{"name": "더미업체", "category": "스튜디오"}]

conn = None
if all([MYSQL_HOST, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DB]):
    try:
        conn = mysql.connector.connect(
            host=MYSQL_HOST, user=MYSQL_USER,
            password=MYSQL_PASSWORD, database=MYSQL_DB,
            port=MYSQL_PORT
        )
        print("MySQL 연결 성공")
    except Exception as e:
        conn = None
        print(f"MySQL 연결 실패: {e}")
else:
    print("MySQL 미설정 - 더미 사용")

def get_user_preference(conn, couple_id):
    if conn:
        try:
            cur = conn.cursor(dictionary=True)
            cur.execute("SELECT * FROM couple_preferences WHERE couple_id = %s", (couple_id,))
            row = cur.fetchone(); cur.close()
            if row:
                return row
        except:
            pass
    return DUMMY_PREF

def get_user_likes(conn, couple_id):
    if conn:
        try:
            cur = conn.cursor(dictionary=True)
            cur.execute("SELECT * FROM couple_venue_likes WHERE couple_id = %s", (couple_id,))
            rows = cur.fetchall(); cur.close()
            if rows:
                return rows
        except:
            pass
    return DUMMY_LIKES

print("사용자 함수 정의 완료")

MySQL 연결 성공
사용자 함수 정의 완료


## 3. Neo4j 스키마 추출

In [7]:
def get_schema(uri, user, password):
    d = GraphDatabase.driver(uri, auth=basic_auth(user, password))
    with d.session() as session:
        nodes = session.run("MATCH (n) WITH DISTINCT labels(n) AS lbls, keys(n) AS ks, n UNWIND lbls AS l UNWIND ks AS k RETURN l, k, n[k] AS sv")
        rels = session.run("MATCH (a)-[r]->(b) RETURN DISTINCT labels(a) AS sl, type(r) AS rt, labels(b) AS el")
        schema = {"nodes": {}, "relations": []}
        for rec in nodes:
            l, k = rec["l"], rec["k"]
            if l not in schema["nodes"]: schema["nodes"][l] = {}
            v = rec["sv"]
            t = "STRING" if isinstance(v, str) else "INTEGER" if isinstance(v, int) else "FLOAT" if isinstance(v, float) else "UNKNOWN"
            schema["nodes"][l][k] = t
        for rec in rels:
            sl = rec["sl"][0] if rec["sl"] else "?"
            el = rec["el"][0] if rec["el"] else "?"
            schema["relations"].append(f"(:{sl})-[:{rec['rt']}]->(:{el})")
    d.close()
    return schema

def format_schema(schema):
    lines = ["Node properties:"]
    for label, props in schema["nodes"].items():
        ps = ", ".join(f"{k}: {v}" for k, v in props.items())
        lines.append(f"  {label} {{{ps}}}")
    lines.append("Relationships:")
    for r in schema["relations"]: lines.append(f"  {r}")
    return "\n".join(lines)

schema = get_schema(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
neo4j_schema = format_schema(schema)
print(neo4j_schema)

Node properties:
  Review {name: STRING, date: STRING, score: INTEGER, contents: STRING}
  Vendor {likeCnt: INTEGER, iweddingNo: STRING, productPrice: INTEGER, category: STRING, profile: STRING, eventOptionPrice: INTEGER, orderCnt: INTEGER, detailCmt: STRING, enterpriseCode: STRING, salePrice: INTEGER, holiday: STRING, viewCnt: INTEGER, eventPrice: INTEGER, subCategory: STRING, reviewCnt: INTEGER, profileUrl: STRING, rating: FLOAT, productName: STRING, region: STRING, coverUrl: STRING, typeName: STRING, tel: STRING, partnerId: INTEGER, name: STRING, address: STRING, uuid: STRING}
  Region {name: STRING}
  Tag {category: STRING, name: STRING, typeName: STRING, type: INTEGER}
  StyleFilter {name: STRING, partnerType: INTEGER, id: INTEGER}
  Package {desc: STRING, title: STRING, value: STRING}
  Hall {minRentalPrice: INTEGER, rating: FLOAT, reviewCnt: INTEGER, availableContract: INTEGER, profileUrl: STRING, address: STRING, maxIndividualHallPrice: INTEGER, tel: STRING, region: STRING, nam

## 4. Few-shot Cypher 예시 (3개 카테고리 통합)

스튜디오/드레스/메이크업 예시가 모두 포함되어 있어, GraphRAG가 질문에서 카테고리를 자동 판별합니다.

In [8]:
examples = [
    # ============================================
    # 스튜디오 (category: 'studio')
    # ============================================
    """USER INPUT: '스튜디오 추천해줘'
QUERY:
MATCH (v:Vendor {category:'studio'}) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '200만원 이하 스튜디오'
QUERY:
MATCH (v:Vendor {category:'studio'}) WHERE v.salePrice <= 2000000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '야외씬 잘 찍는곳'
QUERY:
MATCH (v:Vendor {category:'studio'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '야외' OR t.name CONTAINS '로드' OR t.name CONTAINS '가든'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '인물중심 스타일 추천'
QUERY:
MATCH (v:Vendor {category:'studio'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '인물중심'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '강남 스튜디오 150만원 이하'
QUERY:
MATCH (v:Vendor {category:'studio'})-[:IN_REGION]->(r:Region)
WHERE r.name CONTAINS '강남' AND v.salePrice <= 1500000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '역삼역 근처 스튜디오'
QUERY:
MATCH (v:Vendor {category:'studio'})
WHERE v.address CONTAINS '역삼'
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '논현동 메이크업샵'
QUERY:
MATCH (v:Vendor {category:'makeup'})
WHERE v.address CONTAINS '논현'
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '신사동 드레스샵 추천'
QUERY:
MATCH (v:Vendor {category:'dress'})
WHERE v.address CONTAINS '신사'
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '줄리의정원 가격이랑 패키지 알려줘'
QUERY:
MATCH (v:Vendor {category:'studio'}) WHERE v.name CONTAINS '줄리의정원'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_PACKAGE]->(p:Package)
RETURN v.name AS name, v.salePrice AS salePrice, v.productPrice AS originalPrice, v.rating AS rating, v.address AS address,
       collect(DISTINCT t.name) AS tags, collect(DISTINCT {title: p.title, value: p.value}) AS packages""",

    """USER INPUT: '줄리의정원이랑 식스플로어 비교해줘'
QUERY:
MATCH (v:Vendor {category:'studio'})
WHERE v.name CONTAINS '줄리의정원' OR v.name CONTAINS '식스플로어'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, collect(DISTINCT t.name) AS tags, avg(rv.score) AS avgScore, count(rv) AS revCnt
RETURN v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, tags, round(avgScore, 1) AS avgScore, revCnt
ORDER BY v.name""",

    """USER INPUT: '줄리의정원과 비슷한 스타일'
QUERY:
MATCH (v1:Vendor {category:'studio'})-[:HAS_TAG]->(t:Tag)<-[:HAS_TAG]-(v2:Vendor {category:'studio'})
WHERE v1.name CONTAINS '줄리의정원' AND v1 <> v2
WITH v2, collect(DISTINCT t.name) AS sharedTags, count(t) AS cnt
RETURN v2.name AS name, v2.salePrice AS price, v2.rating AS rating, v2.address AS address, sharedTags
ORDER BY cnt DESC, v2.rating DESC LIMIT 5""",

    """USER INPUT: '야외씬이랑 어울리는 스타일은?'
QUERY:
MATCH (t1:Tag {category:'studio'})-[c:CO_OCCURS]-(t2:Tag {category:'studio'})
WHERE t1.name CONTAINS '야외'
RETURN t2.name AS relatedTag, c.count AS freq
ORDER BY freq DESC LIMIT 10""",

    """USER INPUT: '리뷰 좋은 강남 스튜디오'
QUERY:
MATCH (v:Vendor {category:'studio'})-[:IN_REGION]->(r:Region)
WHERE r.name CONTAINS '강남'
MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, avg(rv.score) AS avgScore, count(rv) AS revCnt
WHERE revCnt >= 3
RETURN v.name AS name, v.salePrice AS price, v.address AS address, round(avgScore, 1) AS avgScore, revCnt
ORDER BY avgScore DESC LIMIT 10""",

    # ============================================
    # 드레스 (category: 'dress')
    # ============================================
    """USER INPUT: '드레스 추천해줘'
QUERY:
MATCH (v:Vendor {category:'dress'}) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '200만원 이하 드레스'
QUERY:
MATCH (v:Vendor {category:'dress'}) WHERE v.salePrice <= 2000000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '촬영+본식 드레스 4벌 이상'
QUERY:
MATCH (v:Vendor {category:'dress'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '4벌' OR t.name CONTAINS '5벌'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '컬러 드레스 있는 곳'
QUERY:
MATCH (v:Vendor {category:'dress'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '컬러'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '셀렉션H 가격이랑 구성 알려줘'
QUERY:
MATCH (v:Vendor {category:'dress'}) WHERE v.name CONTAINS '셀렉션'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_PACKAGE]->(p:Package)
RETURN v.name AS name, v.salePrice AS salePrice, v.productPrice AS originalPrice, v.rating AS rating, v.address AS address,
       collect(DISTINCT t.name) AS tags, collect(DISTINCT {title: p.title, value: p.value}) AS packages""",

    """USER INPUT: '셀렉션H와 비슷한 스타일 드레스샵'
QUERY:
MATCH (v1:Vendor {category:'dress'})-[:HAS_TAG]->(t:Tag)<-[:HAS_TAG]-(v2:Vendor {category:'dress'})
WHERE v1.name CONTAINS '셀렉션' AND v1 <> v2
WITH v2, collect(DISTINCT t.name) AS sharedTags, count(t) AS cnt
RETURN v2.name AS name, v2.salePrice AS price, v2.rating AS rating, v2.address AS address, sharedTags
ORDER BY cnt DESC, v2.rating DESC LIMIT 5""",

    """USER INPUT: '리뷰 좋은 강남 드레스샵'
QUERY:
MATCH (v:Vendor {category:'dress'})-[:IN_REGION]->(r:Region)
WHERE r.name CONTAINS '강남'
MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, avg(rv.score) AS avgScore, count(rv) AS revCnt
WHERE revCnt >= 3
RETURN v.name AS name, v.salePrice AS price, v.address AS address, round(avgScore, 1) AS avgScore, revCnt
ORDER BY avgScore DESC LIMIT 10""",

    # ============================================
    # 메이크업 (category: 'makeup')
    # ============================================
    """USER INPUT: '메이크업 추천해줘'
QUERY:
MATCH (v:Vendor {category:'makeup'}) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '50만원 이하 메이크업'
QUERY:
MATCH (v:Vendor {category:'makeup'}) WHERE v.salePrice <= 500000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '내추럴 메이크업 추천'
QUERY:
MATCH (v:Vendor {category:'makeup'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '내추럴' OR t.name CONTAINS '깨끗'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '실장급 메이크업'
QUERY:
MATCH (v:Vendor {category:'makeup'})-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '실장'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '르보청담 가격이랑 구성 알려줘'
QUERY:
MATCH (v:Vendor {category:'makeup'}) WHERE v.name CONTAINS '르보'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_PACKAGE]->(p:Package)
RETURN v.name AS name, v.salePrice AS salePrice, v.productPrice AS originalPrice, v.rating AS rating, v.address AS address,
       collect(DISTINCT t.name) AS tags, collect(DISTINCT {title: p.title, value: p.value}) AS packages""",

    """USER INPUT: '르보청담과 비슷한 스타일 메이크업샵'
QUERY:
MATCH (v1:Vendor {category:'makeup'})-[:HAS_TAG]->(t:Tag)<-[:HAS_TAG]-(v2:Vendor {category:'makeup'})
WHERE v1.name CONTAINS '르보' AND v1 <> v2
WITH v2, collect(DISTINCT t.name) AS sharedTags, count(t) AS cnt
RETURN v2.name AS name, v2.salePrice AS price, v2.rating AS rating, v2.address AS address, sharedTags
ORDER BY cnt DESC, v2.rating DESC LIMIT 5""",

    """USER INPUT: '리뷰 좋은 강남 메이크업샵'
QUERY:
MATCH (v:Vendor {category:'makeup'})-[:IN_REGION]->(r:Region)
WHERE r.name CONTAINS '강남'
MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, avg(rv.score) AS avgScore, count(rv) AS revCnt
WHERE revCnt >= 3
RETURN v.name AS name, v.salePrice AS price, v.address AS address, round(avgScore, 1) AS avgScore, revCnt
ORDER BY avgScore DESC LIMIT 10""",

    # ============================================
    # 공통 패턴
    # ============================================
    """USER INPUT: '요즘 인기있는 곳'
QUERY:
MATCH (v:Vendor) WHERE v.orderCnt > 0
RETURN v.partnerId AS id, v.name AS name, v.category AS category, v.salePrice AS price, v.rating AS rating, v.address AS address, v.orderCnt AS orders, v.profileUrl AS url
ORDER BY v.orderCnt DESC LIMIT 10""",

    """USER INPUT: '가장 저렴한 곳 5개'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.category AS category, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.salePrice ASC LIMIT 5""",

    # ============================================
    # 복합 질문 (UNION 사용 금지 — 하나의 MATCH로 처리)
    # ============================================
    """USER INPUT: '로이스튜디오는 어디에 있고 근처에 다른 스튜디오도 추천해줘'
QUERY:
MATCH (v1:Vendor {category:'studio'})-[:IN_REGION]->(r:Region)<-[:IN_REGION]-(v2:Vendor {category:'studio'})
WHERE v1.name CONTAINS '로이스튜디오'
RETURN v1.name AS name, v1.address AS address, v1.salePrice AS price, v1.rating AS rating, r.name AS region,
       collect(DISTINCT {name: v2.name, price: v2.salePrice, rating: v2.rating})[..5] AS nearbyVendors""",

    """USER INPUT: '줄리의정원 정보랑 비슷한 가격대 다른 곳도 알려줘'
QUERY:
MATCH (v1:Vendor {category:'studio'})-[:IN_REGION]->(r:Region)<-[:IN_REGION]-(v2:Vendor {category:'studio'})
WHERE v1.name CONTAINS '줄리의정원' AND v1 <> v2 AND abs(v2.salePrice - v1.salePrice) < 300000
RETURN v1.name AS name, v1.address AS address, v1.salePrice AS price, v1.rating AS rating, r.name AS region,
       collect(DISTINCT {name: v2.name, price: v2.salePrice, rating: v2.rating})[..5] AS similarPriceVendors""",

    # ============================================
    # 이전 대화 맥락 참조 패턴
    # ============================================
    """USER INPUT: '아까 추천한 업체 중에서 가장 싼 곳'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.category AS category, v.salePrice AS price, v.rating AS rating, v.address AS address, v.profileUrl AS url
ORDER BY v.salePrice ASC LIMIT 5""",

    """USER INPUT: '그 업체 상세 정보 알려줘'
QUERY:
MATCH (v:Vendor) WHERE v.name CONTAINS $vendorName
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (v)-[:HAS_PACKAGE]->(p:Package)
OPTIONAL MATCH (v)-[:HAS_REVIEW]->(rv:Review)
WITH v, collect(DISTINCT t.name) AS tags, collect(DISTINCT {title: p.title, value: p.value}) AS packages, collect(DISTINCT {name: rv.name, score: rv.score, contents: rv.contents})[..5] AS reviews
RETURN v.name AS name, v.category AS category, v.salePrice AS price, v.productPrice AS originalPrice, v.rating AS rating, v.address AS address, v.tel AS tel, tags, packages, reviews""",
]

print(f"Few-shot 예시 {len(examples)}개 로드 완료 (스튜디오/드레스/메이크업 통합)")

Few-shot 예시 33개 로드 완료 (스튜디오/드레스/메이크업 통합)


## 5. LLM / Retriever / GraphRAG

In [ ]:
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings

SDM_RAG_TEMPLATE = RagTemplate(
    template="""당신은 웨딩 스드메(스튜디오/드레스/메이크업) 전문 추천 챗봇입니다.

아래 검색 결과를 기반으로 답변하세요.

[답변 규칙]
1. 검색 결과에 있는 데이터만 사용. 없는 내용을 만들지 마세요.
2. 추천 시 업체명, 가격, 평점, 특징을 포함해 구체적으로 답변.
3. 가격은 만원 단위로 표시 (예: 163만원).
4. 여러 업체를 추천할 때는 번호를 매겨 정리.
5. 비교 질문에는 항목별로 나눠서 비교.
6. conditionScore가 높은(조건 충족 많은) 업체를 우선 추천.
7. 부분 충족 업체는 "조건을 넓히면 이런 곳도 있습니다"로 구분.
8. 데이터가 없으면 "해당 조건의 업체를 찾지 못했습니다"라고 솔직하게 답변.

검색 결과:
{context}

사용자 질문: {query_text}

답변:""",
    expected_inputs=["context", "query_text"]
)

llm = OpenAILLM(model_name="gpt-4o", model_params={"temperature": 0, "max_tokens": 2000})

# --- Text2Cypher (구조적 검색: 가격, 지역, 업체명) ---
text2cypher_retriever = Text2CypherRetriever(driver=driver, llm=llm, neo4j_schema=neo4j_schema, examples=examples)
rag_cypher = GraphRAG(retriever=text2cypher_retriever, llm=llm, prompt_template=SDM_RAG_TEMPLATE)

# --- VectorCypher (의미 검색: 스타일, 분위기, 추상 표현) ---
embedder = OpenAIEmbeddings(model="text-embedding-3-small")

# 지역명 목록 (라우터 + retrieval_query에서 사용)
REGIONS = [
    "강남", "서초", "마포", "성북", "송파", "강서", "강동", "강북",
    "용산", "종로", "중구", "영등포", "동작", "관악", "서대문", "은평",
    "광진", "성동", "동대문", "중랑", "노원", "도봉", "구로", "금천",
    "양천", "잠실", "역삼", "논현", "신사", "청담", "압구정", "삼성",
    "반포", "방배", "잠원", "합정", "상수", "망원", "연남", "홍대",
    "수원", "성남", "분당", "일산", "판교", "광교",
]

CATEGORY_MAP = {
    "스튜디오": "studio", "studio": "studio", "스냅": "studio", "촬영": "studio",
    "드레스": "dress", "dress": "dress",
    "메이크업": "makeup", "makeup": "makeup", "헤어": "makeup",
}

# 의미 시그널 키워드
SEMANTIC_SIGNALS = [
    "럭셔리", "고급", "모던", "클래식", "화려", "심플", "우아",
    "자연스러", "내추럴", "분위기", "감성", "따뜻", "아늑", "깔끔",
    "트렌디", "빈티지", "세련", "프리미엄", "로맨틱", "드라마틱",
    "비슷한", "같은 느낌", "같은 스타일",
]


def build_vector_retrieval_query(category=None, region=None, max_price=None, min_price=None):
    """조건에 따라 동적 retrieval_query 생성 (하드 필터 + 소프트 점수)"""
    parts = ["WITH node AS v, score"]

    # 하드 필터: 카테고리
    if category:
        parts.append(f"WHERE v.category = '{category}'")

    parts.append("MATCH (v)-[:IN_REGION]->(r:Region)")
    parts.append("OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)")
    parts.append("OPTIONAL MATCH (v)-[:HAS_REVIEW]->(rv:Review)")
    parts.append("""WITH v, score, r,
        collect(DISTINCT t.name) AS tags,
        avg(rv.score) AS avgReviewScore,
        count(rv) AS reviewCount""")

    # 소프트 점수
    score_parts = []
    if region:
        score_parts.append(f"CASE WHEN r.name CONTAINS '{region}' THEN 1 ELSE 0 END AS regionMatch")
    if max_price:
        score_parts.append(f"CASE WHEN v.salePrice <= {max_price} AND v.salePrice > 0 THEN 1 ELSE 0 END AS priceMatch")
    if min_price:
        score_parts.append(f"CASE WHEN v.salePrice >= {min_price} THEN 1 ELSE 0 END AS priceMinMatch")

    if score_parts:
        parts.append("WITH v, score, r, tags, avgReviewScore, reviewCount,")
        parts.append(",\n    ".join(score_parts))

        score_names = []
        if region: score_names.append("regionMatch")
        if max_price: score_names.append("priceMatch")
        if min_price: score_names.append("priceMinMatch")
        condition_score = " + ".join(score_names)

        parts.append(f"""RETURN v.name AS name, v.category AS category,
    v.salePrice AS price, v.rating AS rating,
    v.address AS address, v.profileUrl AS url,
    tags, round(avgReviewScore, 1) AS avgReviewScore,
    reviewCount, round(score, 4) AS vectorScore,
    {', '.join(score_names)},
    ({condition_score}) AS conditionScore
ORDER BY conditionScore DESC, score DESC LIMIT 10""")
    else:
        parts.append(f"""RETURN v.name AS name, v.category AS category,
    v.salePrice AS price, v.rating AS rating,
    v.address AS address, v.profileUrl AS url,
    tags, round(avgReviewScore, 1) AS avgReviewScore,
    reviewCount, round(score, 4) AS vectorScore
ORDER BY score DESC LIMIT 10""")

    return "\n".join(parts)


def create_vector_rag(category=None, region=None, max_price=None, min_price=None):
    """조건에 맞는 VectorCypherRetriever + GraphRAG 생성"""
    rq = build_vector_retrieval_query(category, region, max_price, min_price)
    vr = VectorCypherRetriever(
        driver=driver,
        index_name="vendor_embedding_index",
        embedder=embedder,
        retrieval_query=rq,
        result_formatter=None,
    )
    return GraphRAG(retriever=vr, llm=llm, prompt_template=SDM_RAG_TEMPLATE), rq


# 기본 VectorCypher (조건 없는 fallback용)
rag_vector, _ = create_vector_rag()

# 하위 호환용
rag = rag_cypher

print("GraphRAG 초기화 완료")
print(f"  - Text2CypherRetriever (구조 검색)")
print(f"  - VectorCypherRetriever (의미 검색, 동적 retrieval_query)")

## 6. Gradio 챗봇

In [ ]:
CONTEXT_KEYWORDS = ["그 중에서", "아까", "위에서", "방금", "거기", "그곳",
                     "더 알려", "자세히", "다른 건", "그 업체", "첫 번째", "두 번째"]

NO_RESULT_PHRASES = ["찾지 못했습니다", "없습니다", "검색 결과가 없"]


def extract_conditions(msg):
    """질문에서 구조적 조건 추출 (정규식, LLM 호출 없음)"""
    conditions = {}

    # 카테고리
    for keyword, cat in CATEGORY_MAP.items():
        if keyword in msg:
            conditions["category"] = cat
            break

    # 지역
    for region in REGIONS:
        if region in msg:
            conditions["region"] = region
            break

    # 가격
    price_max = re.search(r'(\d+)\s*만\s*원?\s*(이하|미만|까지)', msg)
    price_min = re.search(r'(\d+)\s*만\s*원?\s*(이상|부터|초과)', msg)
    if price_max:
        conditions["max_price"] = int(price_max.group(1)) * 10000
    if price_min:
        conditions["min_price"] = int(price_min.group(1)) * 10000

    return conditions


def has_semantic_signal(msg):
    """의미 시그널 존재 여부"""
    return any(s in msg for s in SEMANTIC_SIGNALS)


def route_query(msg, conditions):
    """질문을 Text2Cypher 또는 VectorCypher로 라우팅"""
    semantic = has_semantic_signal(msg)

    if semantic:
        return "vector"
    else:
        return "text2cypher"


def response_fn(message, chat_history, debug_log):
    chat_history = chat_history or []
    msg = message.strip()
    log_lines = [f"[입력] {msg}", ""]

    if any(x in msg for x in ["내 취향", "내 스타일", "내 선호"]):
        log_lines.append("[분기] MySQL — 취향 조회")
        pref = get_user_preference(conn, COUPLE_ID)
        lines = [f"- {k}: {v}" for k, v in pref.items() if k != "couple_id" and v]
        answer = "현재 저장된 취향 정보입니다.\n" + "\n".join(lines)
        log_lines.append("[결과] 취향 정보 반환")

    elif any(x in msg for x in ["좋아요한", "찜한", "찜 목록"]):
        log_lines.append("[분기] MySQL — 찜 목록 조회")
        likes = get_user_likes(conn, COUPLE_ID)
        if likes:
            lines = [f"- {l.get('name','알수없음')} ({l.get('category','')})" for l in likes]
            answer = f"좋아요한 업체가 {len(likes)}건 있습니다.\n" + "\n".join(lines)
        else:
            answer = "좋아요한 업체가 없습니다."
        log_lines.append(f"[결과] {len(likes)}건")

    else:
        refers_previous = any(x in msg for x in CONTEXT_KEYWORDS)
        if refers_previous and chat_history:
            log_lines.append("[맥락] 이전 대화 참조 감지")
            last_answer = ""
            for m in reversed(chat_history):
                if m["role"] == "assistant":
                    last_answer = m["content"][:500]
                    break
            query = f"이전 추천 결과: {last_answer}\n\n후속 질문: {msg}"
        else:
            query = msg

        # 조건 추출 + 라우팅
        conditions = extract_conditions(msg)
        route = route_query(msg, conditions)
        log_lines.append(f"[조건] {conditions if conditions else '없음'}")
        log_lines.append(f"[라우팅] {route}")

        answer = None

        if route == "vector":
            # VectorCypher: 동적 retrieval_query 생성
            log_lines.append(f"[검색] VectorCypherRetriever")
            try:
                vec_rag, rq = create_vector_rag(
                    category=conditions.get("category"),
                    region=conditions.get("region"),
                    max_price=conditions.get("max_price"),
                    min_price=conditions.get("min_price"),
                )
                # 조건이 있으면 로그에 표시
                if any(k in conditions for k in ["region", "max_price", "min_price"]):
                    log_lines.append(f"[점수필터] region={conditions.get('region')}, "
                                     f"maxPrice={conditions.get('max_price')}, "
                                     f"minPrice={conditions.get('min_price')}")

                t0 = time.time()
                result = vec_rag.search(query_text=msg)
                elapsed = time.time() - t0
                answer = result.answer

                if hasattr(result, 'retriever_result') and result.retriever_result:
                    items = getattr(result.retriever_result, 'items', [])
                    log_lines.append(f"[결과] 벡터 유사도 상위 {len(items)}건, {elapsed:.1f}초")
                else:
                    log_lines.append(f"[결과] {elapsed:.1f}초")

            except Exception as e:
                log_lines.append(f"[Vector 오류] {type(e).__name__}: {e}")

        if route == "text2cypher" or answer is None:
            # Text2Cypher (기본 경로 또는 Vector 실패 시 fallback)
            if answer is None and route == "vector":
                log_lines.append("[fallback] Text2Cypher로 전환")
            log_lines.append(f"[검색] Text2CypherRetriever")
            log_lines.append(f"[쿼리] {query[:100]}{'...' if len(query) > 100 else ''}")
            try:
                t0 = time.time()
                result = rag_cypher.search(query_text=query)
                elapsed = time.time() - t0
                answer = result.answer

                if hasattr(result, 'retriever_result') and result.retriever_result:
                    rr = result.retriever_result
                    if hasattr(rr, 'metadata') and rr.metadata and 'cypher' in rr.metadata:
                        log_lines.append(f"[Cypher]\n{rr.metadata['cypher']}")
                    items = getattr(rr, 'items', [])
                    log_lines.append(f"[결과] {len(items)}건, {elapsed:.1f}초")

                # Text2Cypher도 결과 없으면 Vector fallback
                if answer and any(p in answer for p in NO_RESULT_PHRASES) and route != "vector":
                    log_lines.append("[판별] 결과 없음 → VectorCypher fallback")
                    try:
                        vec_rag, _ = create_vector_rag(
                            category=conditions.get("category"),
                        )
                        t0 = time.time()
                        result = vec_rag.search(query_text=msg)
                        elapsed = time.time() - t0
                        answer = result.answer
                        if hasattr(result, 'retriever_result') and result.retriever_result:
                            items = getattr(result.retriever_result, 'items', [])
                            log_lines.append(f"[결과] 벡터 유사도 상위 {len(items)}건, {elapsed:.1f}초")
                    except Exception as e2:
                        log_lines.append(f"[Vector fallback 오류] {type(e2).__name__}: {e2}")

            except Exception as e:
                log_lines.append(f"[Text2Cypher 오류] {type(e).__name__}: {e}")
                answer = "죄송합니다. 처리 중 오류가 발생했습니다. 다시 시도해주세요."

    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": answer})

    new_log = "\n".join(log_lines)
    full_log = f"{debug_log}\n{'─'*40}\n{new_log}" if debug_log else new_log
    return "", chat_history, full_log


with gr.Blocks(title="웨딩 스드메 추천 챗봇", css="footer {display:none}") as demo:
    gr.Markdown("# 웨딩 스드메 추천 챗봇\n스튜디오 / 드레스 / 메이크업 추천을 한 곳에서! (서울/경기)")

    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("### 처리 흐름")
            debug_box = gr.Textbox(
                label="",
                lines=30,
                max_lines=50,
                interactive=False,
            )
            clear_debug_btn = gr.Button("로그 초기화", size="sm")
            clear_debug_btn.click(lambda: "", outputs=[debug_box])

        with gr.Column(scale=7):
            chatbot = gr.Chatbot(height=500)
            with gr.Row():
                msg = gr.Textbox(placeholder="예: 200만원 이하 야외씬 스튜디오 추천해줘", show_label=False, scale=9)
                send_btn = gr.Button("전송", scale=1)
            gr.Markdown("### 이런 질문을 해보세요")
            with gr.Row():
                gr.Button("스튜디오 추천").click(lambda: ("스튜디오 추천해줘", None), outputs=[msg, chatbot])
                gr.Button("드레스 추천").click(lambda: ("드레스 추천해줘", None), outputs=[msg, chatbot])
                gr.Button("메이크업 추천").click(lambda: ("메이크업 추천해줘", None), outputs=[msg, chatbot])
                gr.Button("야외씬 잘 찍는곳").click(lambda: ("야외씬 잘 찍는곳 추천해줘", None), outputs=[msg, chatbot])
            with gr.Row():
                gr.Button("촬영+본식 드레스").click(lambda: ("촬영+본식 드레스 4벌 이상 추천", None), outputs=[msg, chatbot])
                gr.Button("내추럴 메이크업").click(lambda: ("내추럴 메이크업 추천해줘", None), outputs=[msg, chatbot])
                gr.Button("리뷰 좋은 곳").click(lambda: ("리뷰 좋은 곳 추천해줘", None), outputs=[msg, chatbot])
                gr.Button("내 취향 보기").click(lambda: ("내 취향 알려줘", None), outputs=[msg, chatbot])

    msg.submit(response_fn, [msg, chatbot, debug_box], [msg, chatbot, debug_box])
    send_btn.click(response_fn, [msg, chatbot, debug_box], [msg, chatbot, debug_box])

demo.launch(share=True)